In [ ]:
import time
from ugot import ugot
got = ugot.UGOT()
got.initialize('192.168.1.202')
got.open_camera()
PI_CAMERA = 0 # index for laptop / raspbery pi camera

import ultralytics
from ultralytics import YOLO
import cv2
import numpy as np
from IPython.display import Image, display, clear_output

ultralytics.checks()
model = YOLO("yolo26n.pt")
print("Model is ready.")

In [ ]:
# YOLO test: Raspberry Pi camera, no robot movement
camera = cv2.VideoCapture(PI_CAMERA)
if isinstance(PI_CAMERA, int):
    camera.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*'MJPG'))

if not camera.isOpened():
    print("The camera is not opened.")
else:
    print("The camera is on")
    handle = None
    try:
        while True:
            frame_ok, frame = camera.read()
            if not frame_ok:
                continue

            results = model(frame, verbose=False)
            for box in results[0].boxes:
                name = model.names[int(box.cls[0])]
                conf = float(box.conf[0])
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame, f"{name} {conf:.2f}", (x1, max(y1 - 10, 20)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

            jpeg_ok, jpeg = cv2.imencode(".jpg", frame)
            if jpeg_ok:
                if handle is None:
                    handle = display(Image(data=jpeg.tobytes()), display_id=True)
                else:
                    handle.update(Image(data=jpeg.tobytes()))
    except KeyboardInterrupt:
        pass
    finally:
        camera.release()
        print("Test finished.")

In [ ]:
def object_recognition_laptop(target):
    # open the Raspberry Pi camera
    camera = cv2.VideoCapture(PI_CAMERA)
    if isinstance(PI_CAMERA, int):
        camera.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*'MJPG'))
    if not camera.isOpened():
        print("The camera is not opened.")
        return

    print("The camera is on")
    time.sleep(1)
    handle = None
    object_found = None
    try:
        while object_found is None or time.time() - object_found < 3:
            frame_ok, frame = camera.read()
            if not frame_ok:
                continue
            # YOLO block
            results = model(frame, verbose=False)
            for box in results[0].boxes:
                name = model.names[int(box.cls[0])]
                if name == target:
                    if object_found is None:
                        object_found = time.time()
                    conf = float(box.conf[0])
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    cv2.putText(frame, f"{name} {conf:.2f}", (x1, y1 - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            # handle block
            jpeg_ok, jpeg = cv2.imencode(".jpg", frame)
            if jpeg_ok:
                if handle is None:  # create the display area for the first time
                    handle = display(Image(data=jpeg.tobytes()), display_id=True)
                else:  # update the same display area
                    handle.update(Image(data=jpeg.tobytes()))
    finally:
        camera.release()
# 2        
def curve_approach_stop(target):
    import time
    got.open_camera()
    print("The camera is on.")
    time.sleep(1)

    while True:
        picture_raw = got.read_camera_data()
        if picture_raw is None:
            continue
        
        frame = cv2.imdecode(np.frombuffer(picture_raw,np.uint8),cv2.IMREAD_COLOR)
        # YOLO block
        results = model(frame,verbose = False)
        cx = None
        width = None # flag
        for box in results[0].boxes:
            name = model.names[int(box.cls[0])]
            conf = float(box.conf[0])
            if name == target:
                x1,y1,x2,y2 = map(int,box.xyxy[0])
                cx =(x1+x2)//2
                cy =(y1+y2)//2
                width = x2-x1
                cv2.rectangle(frame,(x1,y1),(x2,y2),(0,255,0),2)
                cv2.circle(frame,(cx,cy),2,(0,255,255),2)
                cv2.putText(frame,f"{name} {conf:.2f} cx:{cx}width:{width}",(x1,y1-10),cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,255,0),2)
        
        ok,picture_jpg = cv2.imencode(".jpg",frame)
        display(Image(picture_jpg.tobytes()))
        clear_output(True)
        
        if cx is None:
            got.mecanum_move_xyz(0,10,0)
        elif width>120 and 290<cx<360:
            got.mecanum_stop()
            break
        else:
            got.mecanum_move_xyz(0,10,(320-cx)//4)
# 3            
def patrol_recognize (target):
    import time
    got.open_camera()
    print("The camera is on.")
    time.sleep(1)

    while True:
        picture_raw = got.read_camera_data()
        if picture_raw is None:
            continue
        
        frame = cv2.imdecode(np.frombuffer(picture_raw,np.uint8),cv2.IMREAD_COLOR)
        # YOLO block
        results = model(frame,verbose = False)
        cx = None # flag
        for box in results[0].boxes:
            name = model.names[int(box.cls[0])]
            conf = float(box.conf[0])
            if name == target:
                x1,y1,x2,y2 = map(int,box.xyxy[0])
                cx =(x1+x2)//2
                cy =(y1+y2)//2
                cv2.rectangle(frame,(x1,y1),(x2,y2),(0,255,0),2)
                cv2.circle(frame,(cx,cy),2,(0,255,255),2)
                cv2.putText(frame,f"{name} {conf:.2f} cx:{cx}",(x1,y1-10),cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,255,0),2)

        ok,picture_jpg = cv2.imencode(".jpg",frame)
        display(Image(picture_jpg.tobytes()))
        clear_output(True)
        
        if cx:
            time.sleep(1)
            got.mecanum_stop()
            break
        got.mecanum_move_xyz(0,10,30)

def laptop_YOLO():
    import time
    # open the laptop camera
    camera = cv2.VideoCapture(0,cv2.CAP_AVFOUNDATION)  #Macbook
    #Windows: camera = cv2.VideoCapture(0,cv2.CAP_DSHOW)
    if not camera.isOpened():
        print("The camera is not opened.")
    else:
        print("The camera is on")
        time.sleep(1)
        handle = None
        start_time = time.time()
        while time.time()-start_time<10:
            frame_ok,frame = camera.read()
            if not frame_ok:
                continue
            # YOLO block
            results = model(frame,verbose = False)
            for box in results[0].boxes:
                name = model.names[int(box.cls[0])]
                conf = float(box.conf[0])
                x1,y1,x2,y2 = map(int,box.xyxy[0])
                cv2.rectangle(frame,(x1,y1),(x2,y2),(0,255,0),2)
                cv2.putText(frame,f"{name} {conf:.2f}",(x1,y1-10),cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,255,0),2)
            # handle block    
            if frame_ok:
                jpeg_ok,jpeg = cv2.imencode(".jpg",frame)
                if jpeg_ok:
                    if handle is None: # create the display area for the first time
                        handle = display(Image(data=jpeg.tobytes()),display_id = True)
                    else: # update the same display area
                        handle.update(Image(data=jpeg.tobytes()))

        camera.release()

# 4 bear approach
def bear_curve_approach_stop(target):
    import time
    got.open_camera()
    print("The camera is on.")
    time.sleep(1)

    while True:
        picture_raw = got.read_camera_data()
        if picture_raw is None:
            continue
        
        frame = cv2.imdecode(np.frombuffer(picture_raw,np.uint8),cv2.IMREAD_COLOR)
        # YOLO block
        results = model(frame,verbose = False)
        cx = None
        width = None # flag
        for box in results[0].boxes:
            name = model.names[int(box.cls[0])]
            conf = float(box.conf[0])
            if name == target:
                x1,y1,x2,y2 = map(int,box.xyxy[0])
                cx =(x1+x2)//2
                cy =(y1+y2)//2
                width = x2-x1
                cv2.rectangle(frame,(x1,y1),(x2,y2),(0,255,0),2)
                cv2.circle(frame,(cx,cy),2,(0,255,255),2)
                cv2.putText(frame,f"{name} {conf:.2f} cx:{cx}width:{width}",(x1,y1-10),cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,255,0),2)
        
        ok,picture_jpg = cv2.imencode(".jpg",frame)
        display(Image(picture_jpg.tobytes()))
        clear_output(True)
        
        if cx is None:
            got.mecanum_move_xyz(0,10,0)
        elif width>300 and 290<cx<360:
            got.mecanum_stop()
            break
        else:
            got.mecanum_move_xyz(0,10,(320-cx)//4)